# Predict Customer Churn — RealMLP Tuned Multi-Seed + Feature Engineering

Neural network approach using **RealMLP** from the RTDL library (Yandex Research).
Key differences from GBDT pipelines:
- Numeric features are **standardized** (zero mean, unit variance) per fold
- Low-cardinality categoricals are **one-hot encoded** (simpler than entity embeddings for this dataset)
- **Optuna** tunes architecture (n_blocks, d_block, dropout) + training (lr, weight_decay, batch_size)
- Multi-seed ensemble (5 seeds × 10 folds = 50 models) for stable predictions
- Same feature engineering as GBDT notebooks (row stats + domain interactions)

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from rtdl_revisiting_models import MLP
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import optuna
from tqdm.auto import tqdm
import warnings
import os
import gc

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Environment Detection ────────────────────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/input')
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = '/kaggle/input/playground-series-s6e3/' if IS_KAGGLE else '../data/'

print(f"Environment : {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Device      : {DEVICE}")
print(f"Data dir    : {DATA_DIR}")
print(f"PyTorch ver : {torch.__version__}")

# ── Experiment Settings ───────────────────────────────────────────────────────
RUN_TUNING     = True    # False → skip Optuna, use PRECOMPUTED_PARAMS
N_TRIALS       = 50      # Optuna trials
N_CV_FOLDS     = 5       # Folds inside each Optuna trial
SEEDS          = [42, 2024, 777, 1337, 999]
N_SPLITS       = 10      # Folds per seed in ensemble
TOTAL_MODELS   = len(SEEDS) * N_SPLITS
MAX_EPOCHS     = 200     # Per fold training
PATIENCE       = 20      # Early stopping patience

# Pre-computed fallback (update after first tuning run)
PRECOMPUTED_PARAMS = {
    'n_blocks':     3,
    'd_block':      256,
    'dropout':      0.15,
    'lr':           1e-3,
    'weight_decay': 1e-5,
    'batch_size':   512,
}

## 1. Feature Engineering

In [ ]:
def engineer_features(df):
    """Row-level statistics + domain interaction features on numeric columns."""
    df = df.copy()
    num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

    # Row aggregations
    df['num_sum']  = df[num_cols].sum(axis=1)
    df['num_mean'] = df[num_cols].mean(axis=1)
    df['num_std']  = df[num_cols].std(axis=1)
    df['num_max']  = df[num_cols].max(axis=1)
    df['num_min']  = df[num_cols].min(axis=1)

    # Domain interactions
    df['Average_Monthly_Actual'] = df['TotalCharges'] / (df['tenure'] + 1e-5)
    df['Monthly_diff']  = df['MonthlyCharges'] - df['Average_Monthly_Actual']
    df['Monthly_ratio'] = df['MonthlyCharges'] / (df['Average_Monthly_Actual'] + 1e-5)

    return df

print("Feature engineering function defined.")

## 2. Load & Preprocess Data

In [ ]:
print(f"Loading data from {DATA_DIR} ...")
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))
sub      = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))

TARGET = 'Churn'
if train_df[TARGET].dtype == 'object':
    train_df[TARGET] = train_df[TARGET].map({'Yes': 1, 'No': 0, '1': 1, '0': 0})

# ── Unified preprocessing on train + test ────────────────────────────────────
train_df['is_train'] = 1
test_df['is_train']  = 0
df = pd.concat([train_df, test_df], ignore_index=True)

features = [c for c in train_df.columns if c not in ['id', TARGET, 'is_train']]
categorical_features = []

print("Label encoding categorical features...")
for col in tqdm(features, desc='Encoding'):
    if df[col].dtype == 'object':
        categorical_features.append(col)
        le = LabelEncoder()
        df[col] = df[col].fillna('Missing')
        df[col] = le.fit_transform(df[col].astype(str))

num_features = [c for c in features if c not in categorical_features]
for col in num_features:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

train_enc = df[df['is_train'] == 1].reset_index(drop=True)
test_enc  = df[df['is_train'] == 0].reset_index(drop=True)

# ── Feature Engineering ───────────────────────────────────────────────────────
print("\nEngineering features...")
X      = engineer_features(train_enc[features])
X_test = engineer_features(test_enc[features])
y      = train_enc[TARGET]

# For NN: one-hot encode low-cardinality categoricals
all_features = list(X.columns)
print(f"\nOne-hot encoding {len(categorical_features)} categorical features...")
X      = pd.get_dummies(X, columns=categorical_features, dtype=float)
X_test = pd.get_dummies(X_test, columns=categorical_features, dtype=float)

# Align columns (in case of category mismatch)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

feature_names = list(X.columns)
D_IN = len(feature_names)

print(f"\nTrain shape : {X.shape}  (d_in={D_IN})")
print(f"Test shape  : {X_test.shape}")
print(f"Target dist : {y.value_counts().to_dict()}")

## 3. Training Utilities

In [ ]:
def train_one_fold(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_tst: np.ndarray,
    params: dict,
    max_epochs: int = MAX_EPOCHS,
    patience: int = PATIENCE,
    verbose: bool = False,
) -> tuple:
    """Train a single RealMLP fold. Returns (val_preds, test_preds, best_auc)."""
    n_blocks    = params['n_blocks']
    d_block     = params['d_block']
    dropout     = params['dropout']
    lr          = params['lr']
    weight_decay = params['weight_decay']
    batch_size  = params['batch_size']

    # Standardize numeric features (fit on train fold only)
    scaler = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_tst_s = scaler.transform(X_tst)

    # Convert to tensors
    X_tr_t  = torch.tensor(X_tr_s,  dtype=torch.float32, device=DEVICE)
    y_tr_t  = torch.tensor(y_tr,    dtype=torch.float32, device=DEVICE).unsqueeze(1)
    X_val_t = torch.tensor(X_val_s, dtype=torch.float32, device=DEVICE)
    y_val_t = torch.tensor(y_val,   dtype=torch.float32, device=DEVICE).unsqueeze(1)
    X_tst_t = torch.tensor(X_tst_s, dtype=torch.float32, device=DEVICE)

    train_ds = TensorDataset(X_tr_t, y_tr_t)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)

    # Model
    d_in = X_tr_s.shape[1]
    model = MLP(d_in=d_in, d_out=1, n_blocks=n_blocks, d_block=d_block, dropout=dropout).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    # Cosine annealing scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_epochs)

    best_auc = 0.0
    best_state = None
    wait = 0

    for epoch in range(max_epochs):
        # Train
        model.train()
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t).squeeze(1).cpu().numpy()
        val_probs = 1 / (1 + np.exp(-val_logits))  # sigmoid
        auc = roc_auc_score(y_val, val_probs)

        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            break

    # Restore best model and predict
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        val_logits  = model(X_val_t).squeeze(1).cpu().numpy()
        test_logits = model(X_tst_t).squeeze(1).cpu().numpy()

    val_preds  = 1 / (1 + np.exp(-val_logits))
    test_preds = 1 / (1 + np.exp(-test_logits))

    del model, optimizer, scheduler, X_tr_t, y_tr_t, X_val_t, y_val_t, X_tst_t
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

    return val_preds, test_preds, best_auc

print("Training utilities defined.")

## 4. Optuna Hyperparameter Tuning

Each trial trains a 5-fold CV to evaluate the hyperparameter set.

In [ ]:
X_np      = X.values.astype(np.float32)
X_test_np = X_test.values.astype(np.float32)
y_np      = y.values.astype(np.float32)

if RUN_TUNING:
    print(f"Starting Optuna tuning: {N_TRIALS} trials x {N_CV_FOLDS}-fold CV each")
    print(f"Device: {DEVICE}, Max epochs per fold: {MAX_EPOCHS}, Patience: {PATIENCE}\n")

    pbar = tqdm(total=N_TRIALS, desc='Optuna Trials', unit='trial')
    best_value_so_far = [0.0]

    def objective(trial):
        params = {
            'n_blocks':     trial.suggest_int('n_blocks', 1, 5),
            'd_block':      trial.suggest_categorical('d_block', [64, 128, 256, 384, 512]),
            'dropout':      trial.suggest_float('dropout', 0.0, 0.5),
            'lr':           trial.suggest_float('lr', 1e-5, 1e-2, log=True),
            'weight_decay': trial.suggest_float('weight_decay', 1e-7, 1e-3, log=True),
            'batch_size':   trial.suggest_categorical('batch_size', [256, 512, 1024, 2048]),
        }

        skf = StratifiedKFold(n_splits=N_CV_FOLDS, shuffle=True, random_state=42)
        fold_aucs = []

        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_np, y_np)):
            _, _, auc = train_one_fold(
                X_np[tr_idx], y_np[tr_idx],
                X_np[val_idx], y_np[val_idx],
                X_test_np,
                params,
                max_epochs=100,   # reduced for tuning speed
                patience=10,
            )
            fold_aucs.append(auc)

            # Prune early if first folds look bad
            trial.report(np.mean(fold_aucs), fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        score = np.mean(fold_aucs)
        if score > best_value_so_far[0]:
            best_value_so_far[0] = score
        pbar.set_postfix({'AUC': f'{score:.5f}', 'Best': f'{best_value_so_far[0]:.5f}'})
        pbar.update(1)
        return score

    study = optuna.create_study(
        direction='maximize',
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1),
    )
    study.optimize(objective, n_trials=N_TRIALS)
    pbar.close()

    best_trial = study.best_trial
    print(f"\n{'='*55}")
    print(f"Best CV AUC  : {best_trial.value:.5f}")
    print(f"Best params  :")
    for k, v in best_trial.params.items():
        print(f"  {k:20s}: {v}")
    print(f"{'='*55}")

    best_params = best_trial.params

    # Save tuning results
    study.trials_dataframe().to_csv('realmlp_tuning_results.csv', index=False)
    print("\nAll trial results saved -> realmlp_tuning_results.csv")

else:
    print("Skipping tuning — using PRECOMPUTED_PARAMS.")
    best_params = PRECOMPUTED_PARAMS
    for k, v in PRECOMPUTED_PARAMS.items():
        print(f"  {k:20s}: {v}")

## 5. Multi-Seed Ensemble

5 seeds x 10 folds = 50 models averaged for stable predictions.

In [ ]:
print(f"Multi-Seed Ensemble: {len(SEEDS)} seeds x {N_SPLITS} folds = {TOTAL_MODELS} models\n")

test_preds_sum = np.zeros(len(X_test_np))
global_oof_sum = np.zeros(len(X_np))
fold_auc_log   = []

outer_bar = tqdm(SEEDS, desc='Seeds', position=0)

for seed in outer_bar:
    outer_bar.set_description(f'Seed {seed}')

    # Set all random seeds
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)

    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros(len(X_np))

    inner_bar = tqdm(
        enumerate(skf.split(X_np, y_np)),
        total=N_SPLITS,
        desc=f'  Folds',
        position=1,
        leave=False,
    )

    for fold, (train_idx, val_idx) in inner_bar:
        val_preds, test_preds, fold_auc = train_one_fold(
            X_np[train_idx], y_np[train_idx],
            X_np[val_idx],   y_np[val_idx],
            X_test_np,
            best_params,
        )

        fold_auc_log.append(fold_auc)
        seed_oof[val_idx]         = val_preds
        global_oof_sum[val_idx]  += val_preds
        test_preds_sum           += test_preds

        inner_bar.set_postfix({'fold_auc': f'{fold_auc:.5f}'})

    seed_auc = roc_auc_score(y_np, seed_oof)
    outer_bar.set_postfix({'seed_oof_auc': f'{seed_auc:.5f}'})

# ── Final metrics ─────────────────────────────────────────────────────────────
global_oof = global_oof_sum / len(SEEDS)
final_auc  = roc_auc_score(y_np, global_oof)

print(f"\n{'='*55}")
print(f"Fold AUC  — mean: {np.mean(fold_auc_log):.5f}  std: {np.std(fold_auc_log):.5f}")
print(f"Global Ensemble OOF AUC : {final_auc:.5f}")
print(f"{'='*55}")

## 6. Generate Submission

In [ ]:
final_test_preds = test_preds_sum / TOTAL_MODELS
sub[TARGET] = final_test_preds

out_file = 'submission_realmlp_tuned_multiseed_fe.csv'
sub.to_csv(out_file, index=False)

print(f"Submission saved -> {out_file}")
print(f"Prediction range: [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]")
sub.head()

## 7. Save OOF for Blending

Save OOF predictions so notebook 08 (or a new blend notebook) can incorporate RealMLP.

In [ ]:
oof_df = pd.DataFrame({'id': train_df['id'], 'oof_pred': global_oof})
oof_df.to_csv('oof_realmlp_tuned_multiseed_fe.csv', index=False)
print(f"OOF predictions saved -> oof_realmlp_tuned_multiseed_fe.csv")
print(f"OOF AUC: {final_auc:.5f}")

## Next Steps

1. **Submit** `submission_realmlp_tuned_multiseed_fe.csv` to get standalone LB score
2. **Blend** with best XGBoost + LightGBM submissions in notebook 08/20
3. **Tune blend weights**: RealMLP provides orthogonal predictions → expect +0.001-0.003 from blending
4. **Try**: Increase `N_TRIALS` to 100, increase `MAX_EPOCHS` to 300 for final run

```bash
kaggle competitions submit -c playground-series-s6e3 \
  -f submission_realmlp_tuned_multiseed_fe.csv \
  -m "RealMLP 50-model ensemble, Optuna tuned, FE"
```